# 02 — Capability Comparison — Practical Examples

**Covers**: Side-by-side reasoning, coding, instruction-following benchmarks on the same prompts, latency comparison

In [ ]:
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. Reasoning Test — Multi-Step Math

In [ ]:
REASONING_PROBLEMS = [
    {
        'id': 'R1',
        'problem': 'A baker makes 3 batches daily. Each batch = 90 minutes. She starts at 6 AM and takes a 30-min lunch after batch 2. What time does she finish batch 3?',
        'correct': '12:00 PM'
    },
    {
        'id': 'R2',
        'problem': 'Alice is 3x older than Bob. In 6 years, Alice will be 2x older than Bob. How old are they now?',
        'correct': 'Alice=18, Bob=6'
    },
    {
        'id': 'R3',
        'problem': 'A train leaves city A at 9 AM going 60 mph. Another leaves city B at 10 AM going 80 mph toward city A. Cities are 280 miles apart. When and where do they meet?',
        'correct': '11 AM, 120 miles from A'
    },
]

for prob in REASONING_PROBLEMS:
    t = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Solve step by step. End your answer with "ANSWER: [final answer]"'},
            {'role': 'user', 'content': prob['problem']}
        ],
        max_tokens=200, temperature=0
    )
    latency = (time.perf_counter() - t) * 1000
    answer = r.choices[0].message.content
    
    # Extract final answer
    final = answer.split('ANSWER:')[-1].strip().split('\n')[0] if 'ANSWER:' in answer else 'NOT FOUND'
    correct = '✅' if prob['correct'].lower().replace(' ','') in final.lower().replace(' ','') else '❌'
    
    print(f"[{prob['id']}] {correct} ({latency:.0f}ms)")
    print(f"  Expected: {prob['correct']}")
    print(f"  Got:      {final[:80]}")
    print()

## 2. Instruction-Following Test — Strict Format Compliance

In [ ]:
import re

IF_TESTS = [
    {
        'id': 'IF1',
        'instruction': 'Reply with ONLY a valid JSON object. No prose, no markdown. Fields: name(str), capital(str), population_millions(float)',
        'question': 'Tell me about France.',
        'checks': ['name', 'capital', 'population_millions']
    },
    {
        'id': 'IF2',
        'instruction': 'Respond with a Python list of exactly 5 items. No explanation. Just: ["item1", "item2", ...]',
        'question': 'Name 5 programming languages.',
        'checks': ['python list', 'exactly 5']
    },
    {
        'id': 'IF3',
        'instruction': 'Use exactly 3 bullet points. Each bullet must start with an emoji. No introduction, no conclusion.',
        'question': 'List 3 benefits of Python.',
        'checks': ['3 bullets', 'emoji']
    },
]

for test in IF_TESTS:
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': test['instruction']},
            {'role': 'user',   'content': test['question']}
        ],
        max_tokens=150, temperature=0
    )
    output = r.choices[0].message.content
    
    # Check compliance
    if test['id'] == 'IF1':
        try:
            parsed = json.loads(output)
            all_fields = all(k in parsed for k in test['checks'])
            compliant = '✅' if all_fields and isinstance(parsed, dict) else '❌'
        except: compliant = '❌'
    elif test['id'] == 'IF2':
        try:
            parsed = json.loads(output.strip())
            compliant = '✅' if isinstance(parsed, list) and len(parsed) == 5 else '❌'
        except: compliant = '❌'
    else:  # IF3
        bullets = [l for l in output.split('\n') if l.strip().startswith(('•', '-', '*')) or (l.strip() and l.strip()[0] in '🌟💡✨🚀🎯⚡🔥')]
        compliant = '✅' if len(bullets) == 3 else '❌'
    
    print(f"[{test['id']}] {compliant} Instruction compliance")
    print(f"  Output: {output[:100]}")
    print()

## 3. Coding Benchmark — Write and Debug

In [ ]:
CODING_TASKS = [
    {
        'id': 'C1',
        'task': 'Write a Python function sliding_window_max(nums: list[int], k: int) -> list[int] that returns the maximum in each sliding window of size k. Time complexity must be O(n).',
    },
    {
        'id': 'C2',
        'task': 'Debug this Python code:\ndef find_duplicates(arr):\n    seen = set()\n    for n in arr:\n        if n in seen:\n            seen.add(n)\n    return list(seen)\n\nThe function is supposed to return duplicate elements only, but it returns nothing. Fix it.',
    },
]

for task in CODING_TASKS:
    t = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Write clean, correct Python code. Include the solution only, with minimal explanation.'},
            {'role': 'user',   'content': task['task']}
        ],
        max_tokens=300, temperature=0
    )
    latency = (time.perf_counter() - t) * 1000
    code = r.choices[0].message.content
    has_def = 'def ' in code
    print(f"[{task['id']}] ({latency:.0f}ms) has function def: {'✅' if has_def else '❌'}")
    print(f"  Code preview: {code[:120]}...")
    print()

## 4. Latency Benchmark — Same Prompt, Measure Speed

In [ ]:
BENCHMARK_PROMPT = 'List 5 Python best practices for writing clean code. Be concise.'
N_RUNS = 3

models_to_bench = ['gpt-4o-mini']  # Add 'gpt-4o' if budget allows

for model_id in models_to_bench:
    latencies, token_counts = [], []
    for run in range(N_RUNS):
        t = time.perf_counter()
        r = client.chat.completions.create(
            model=model_id,
            messages=[{'role': 'user', 'content': BENCHMARK_PROMPT}],
            max_tokens=150
        )
        latency = (time.perf_counter() - t) * 1000
        latencies.append(latency)
        token_counts.append(r.usage.total_tokens)
    
    avg_ms = sum(latencies) / len(latencies)
    avg_tok = sum(token_counts) / len(token_counts)
    tps = avg_tok / (avg_ms / 1000)  # tokens per second
    print(f"[{model_id}]")
    print(f"  Avg latency: {avg_ms:.0f}ms | Avg tokens: {avg_tok:.0f} | Throughput: {tps:.0f} tok/s")
    print(f"  Latencies:   {[f'{l:.0f}ms' for l in latencies]}")

## 📌 Summary

- **Reasoning**: test with multi-step math problems, chain-of-thought required
- **Instruction-following**: strict format compliance matters for agentic pipelines
- **Coding**: write + debug — mini models often 80% as good as flagships
- **Latency**: measure end-to-end latency, not just TTFT — output length matters
- **Run N=3+ benchmark iterations** — single-run results have high variance
- **gpt-4o-mini often good enough** — only pay for flagship where measurably needed